In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
DATA_PATH = os.path.join("../data/", "santander-customer-satisfaction")

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))

print(f"Train 데이터 크기: {train_df.shape}")

train_df.head()

Train 데이터 크기: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [3]:
# 정답 라벨 분리 (맨 마지막 컬럼 TARGET)
y_labels = train_df.iloc[:, -1].copy()

# ID와 TARGET을 제외한 순수 피처 세트 분리
X_features = train_df.drop(columns=['ID', 'TARGET']).copy()

print(f"X_features Shape: {X_features.shape}")
print(f"y_labels Shape: {y_labels.shape}")

X_features Shape: (76020, 369)
y_labels Shape: (76020,)


In [4]:
# var3 결측성 이상치(-999999)를 최빈값(2)으로 치환
X_features['var3'] = X_features['var3'].replace(-999999, 2)

print("var3 이상치 치환 완료 (-999999 잔여 개수):", (X_features['var3'] == -999999).sum())

var3 이상치 치환 완료 (-999999 잔여 개수): 0


In [5]:
# var38 자산 왜도 완화를 위한 로그 변환
X_features['var38'] = np.log1p(X_features['var38'])

print("var38 로그 변환 완료")

var38 로그 변환 완료


In [6]:
# 고객별 0의 개수 카운트
X_features['n0'] = (X_features == 0).sum(axis=1)

# 고객별 거래/잔액의 표준편차
X_features['row_std'] = X_features.std(axis=1)

print("행 통계량 파생 변수(n0, row_std) 생성 완료")

행 통계량 파생 변수(n0, row_std) 생성 완료


In [7]:
# 1. 분산 0인 상수 컬럼 제거 <- 제거 한 것과 안 한 것 둘 다 결과 같음
zero_var_cols = [col for col in X_features.columns if X_features[col].nunique() == 1]
X_features.drop(columns=zero_var_cols, inplace=True)
print(f"1) 제거된 상수 컬럼 수: {len(zero_var_cols)}")

# 2. 중복 컬럼 초고속 제거
dup_cols = X_features.T.duplicated()
dup_col_names = X_features.columns[dup_cols].tolist()
X_features.drop(columns=dup_col_names, inplace=True)
print(f"2) 제거된 중복 컬럼 수: {len(dup_col_names)}")

# 3. var6 유사 피처 및 다중공선성 delta 컬럼 제거
manual_remove = [c for c in X_features.columns if 'var6' in c] + [
    'delta_imp_reemb_var13_1y3', 'delta_imp_reemb_var17_1y3', 
    'delta_imp_trasp_var17_in_1y3', 'delta_imp_trasp_var33_in_1y3'
]
manual_remove = [c for c in manual_remove if c in X_features.columns]
X_features.drop(columns=manual_remove, inplace=True)
print(f"3) 추가 제거된 유사/노이즈 컬럼 수: {len(manual_remove)}")


1) 제거된 상수 컬럼 수: 34
2) 제거된 중복 컬럼 수: 29
3) 추가 제거된 유사/노이즈 컬럼 수: 9


In [8]:
# 1차 분할: 학습 세트(80%)와 최종 테스트 세트(20%)로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels,
    test_size=0.2, 
    stratify=y_labels, 
    random_state=0
)

# 2차 분할: 학습 세트를 다시 훈련용(70%)과 조기 중단 감시용(30%)으로 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.3, 
    stratify=y_train, 
    random_state=0
)

print(f"훈련 세트(X_tr) Shape: {X_tr.shape}")
print(f"검증 세트(X_val) Shape: {X_val.shape}")
print(f"테스트 세트(X_test) Shape: {X_test.shape}")

훈련 세트(X_tr) Shape: (42571, 299)
검증 세트(X_val) Shape: (18245, 299)
테스트 세트(X_test) Shape: (15204, 299)


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
# (1) 훈련 데이터(X_tr)로만 베이스 모델 학습하여 피처 중요도 측정
base_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=156,
    n_jobs=-1
)
base_xgb.fit(X_tr, y_tr)




# (2) 중요도가 0인 피처 확인 후 3개 세트(X_tr, X_val, X_test)에서 동일하게 제거
feat_imp = pd.Series(base_xgb.feature_importances_, index=X_tr.columns)
zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()

X_tr = X_tr.drop(columns=zero_imp_cols)
X_val = X_val.drop(columns=zero_imp_cols)
X_test = X_test.drop(columns=zero_imp_cols)
print(f"제거된 중요도 0인 컬럼 수: {len(zero_imp_cols)}")



# (3) X_tr 기준으로 StandardScaler 학습(fit) 후 변환(transform)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# (4) X_tr 기준으로 PCA 학습(fit) 후 주성분 2개 변환(transform)
pca = PCA(n_components=5, random_state=156)
pca_tr = pca.fit_transform(X_tr_scaled)
pca_val = pca.transform(X_val_scaled)
pca_test = pca.transform(X_test_scaled)

# (5) 각 세트에 pca1, pca2 컬럼 추가
X_tr['pca1'] = pca_tr[:, 0]
X_tr['pca2'] = pca_tr[:, 1]
X_tr['pca3'] = pca_tr[:, 2]
X_tr['pca4'] = pca_tr[:, 3]
X_tr['pca5'] = pca_tr[:, 4]

X_val['pca1'] = pca_val[:, 0]
X_val['pca2'] = pca_val[:, 1]
X_val['pca3'] = pca_val[:, 2]
X_val['pca4'] = pca_val[:, 3]
X_val['pca5'] = pca_val[:, 4]

X_test['pca1'] = pca_test[:, 0]
X_test['pca2'] = pca_test[:, 1]
X_test['pca3'] = pca_test[:, 2]
X_test['pca4'] = pca_test[:, 3]
X_test['pca5'] = pca_test[:, 4]

print(f"훈련 세트(X_tr) 최종 Shape: {X_tr.shape}")
print(f"검증 세트(X_val) 최종 Shape: {X_val.shape}")
print(f"테스트 세트(X_test) 최종 Shape: {X_test.shape}")



제거된 중요도 0인 컬럼 수: 188
훈련 세트(X_tr) 최종 Shape: (42571, 116)
검증 세트(X_val) 최종 Shape: (18245, 116)
테스트 세트(X_test) 최종 Shape: (15204, 116)


In [10]:
# 모델 정의
xgb_clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    eval_metric='auc',
    early_stopping_rounds=100,
    random_state=156,
    n_jobs=-1
)

# 조기 중단(Early Stopping)을 적용한 학습
xgb_clf.fit(
    X_tr, y_tr, 
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    verbose=False
)

# 최종 테스트 세트 점수 산출
test_preds = xgb_clf.predict_proba(X_test)[:, 1]
xgb_roc_score = roc_auc_score(y_test, test_preds)

print("=" * 40)
print(f"XGBoost 최종 테스트 세트 ROC-AUC: {xgb_roc_score:.4f}")
print("=" * 40)

XGBoost 최종 테스트 세트 ROC-AUC: 0.8240


In [ ]:
# hyperopt 하이퍼 파라미터 튜닝

from hyperopt import hp


# max_depth는 5에서 15까지 1간격으로, min_child_weight는 1에서 6까지 1간격으로
# colsample_bytree는 0.5에서 0.95사이, learning_rate는 0.01에서 0.2사이 정규 분포된 값으로 검색.


xgb_search_space = {'max_depth': hp.quniform('max_depth', 4, 15, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 6, 1),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.95),
                    'subsample': hp.uniform('subsample', 0.6, 1.0),
                    'gamma': hp.uniform('gamma', 0, 5),
                    'reg_alpha': hp.uniform('reg_alpha', 0, 2),
                    'reg_lambda': hp.uniform('reg_lambda', 0.5, 5),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
}


In [12]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


# 목적 함수 설정.
# 추후 fmin()에서 입력된 search_space값으로 XGBClassifier 교차 검증 학습 후 -1* roc_auc 평균 값을 반환.  
def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            colsample_bytree=search_space['colsample_bytree'],
                            subsample=search_space['subsample'],
                            gamma=search_space['gamma'],
                            reg_alpha=search_space['reg_alpha'],
                            reg_lambda=search_space['reg_lambda'],
                            early_stopping_rounds=50, eval_metric='auc'
                           )
    # missing=9999999999,
    #             max_depth = 5,
    #             n_estimators=1000,
    #             learning_rate=0.1, 
    #             nthread=4,
    #             subsample=1.0,
    #             colsample_bytree=0.5,
    #             min_child_weight = 3,
    #             scale_pos_weight = ratio,
    #             reg_alpha=0.03,
    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list= []
   
    # 3개 k-fold방식 적용
    kf = KFold(n_splits=5)
    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]
        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
        xgb_clf.fit(X_tr, y_tr, 
                   eval_set=[(X_tr, y_tr), (X_val, y_val)])
   
        # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
        score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)
       
    # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)


In [24]:
from hyperopt import fmin, tpe, Trials


trials = Trials()


# fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=200, # 최대 반복 횟수를 지정합니다.
            trials=trials, rstate=np.random.default_rng(seed=30))


print('best:', best)

[0]	validation_0-auc:0.83094	validation_1-auc:0.81610  
[1]	validation_0-auc:0.83403	validation_1-auc:0.80952  
[2]	validation_0-auc:0.84949	validation_1-auc:0.82528  
[3]	validation_0-auc:0.85199	validation_1-auc:0.82707  
[4]	validation_0-auc:0.85630	validation_1-auc:0.83161  
[5]	validation_0-auc:0.85881	validation_1-auc:0.83379  
[6]	validation_0-auc:0.86122	validation_1-auc:0.83336  
[7]	validation_0-auc:0.86307	validation_1-auc:0.83248  
[8]	validation_0-auc:0.86475	validation_1-auc:0.83536  
[9]	validation_0-auc:0.86552	validation_1-auc:0.83788  
[10]	validation_0-auc:0.86753	validation_1-auc:0.83806 
[11]	validation_0-auc:0.87079	validation_1-auc:0.83846 
[12]	validation_0-auc:0.87128	validation_1-auc:0.83865 
[13]	validation_0-auc:0.87190	validation_1-auc:0.83833 
[14]	validation_0-auc:0.87333	validation_1-auc:0.83864 
[15]	validation_0-auc:0.87433	validation_1-auc:0.83835 
[16]	validation_0-auc:0.87529	validation_1-auc:0.83923 
[17]	validation_0-auc:0.87656	validation_1-auc:0

In [22]:
# n_estimators를 500증가 후 최적으로 찾은 하이퍼 파라미터를 기반으로 학습과 예측 수행.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=round(best['learning_rate'], 5),
                        max_depth=int(best['max_depth']), min_child_weight=int(best['min_child_weight']),
                        early_stopping_rounds=100, eval_metric="auc",gamma=best['gamma'],subsample=best['subsample'],
                        reg_alpha=best['reg_alpha'],reg_lambda=best['reg_lambda'],
                        colsample_bytree=round(best['colsample_bytree'], 5)  
                       )


# evaluation metric을 auc로, early stopping은 100 으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)])


xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:,1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))

[0]	validation_0-auc:0.82897	validation_1-auc:0.82279
[1]	validation_0-auc:0.84020	validation_1-auc:0.83072
[2]	validation_0-auc:0.84024	validation_1-auc:0.83125
[3]	validation_0-auc:0.84424	validation_1-auc:0.83357
[4]	validation_0-auc:0.84478	validation_1-auc:0.83418
[5]	validation_0-auc:0.84780	validation_1-auc:0.83681
[6]	validation_0-auc:0.85077	validation_1-auc:0.83932
[7]	validation_0-auc:0.85251	validation_1-auc:0.84044
[8]	validation_0-auc:0.85387	validation_1-auc:0.84127
[9]	validation_0-auc:0.85520	validation_1-auc:0.84166
[10]	validation_0-auc:0.85606	validation_1-auc:0.84231
[11]	validation_0-auc:0.85619	validation_1-auc:0.84252
[12]	validation_0-auc:0.85673	validation_1-auc:0.84249
[13]	validation_0-auc:0.85775	validation_1-auc:0.84282
[14]	validation_0-auc:0.85898	validation_1-auc:0.84319
[15]	validation_0-auc:0.85961	validation_1-auc:0.84404
[16]	validation_0-auc:0.86050	validation_1-auc:0.84413
[17]	validation_0-auc:0.86117	validation_1-auc:0.84456
[18]	validation_0-au

In [22]:
# n_estimators를 500증가 후 최적으로 찾은 하이퍼 파라미터를 기반으로 학습과 예측 수행.
xgb_clf = XGBClassifier(n_estimators=1000, learning_rate=round(0.1388590465213671, 5),
                        max_depth=4, min_child_weight=6,
                        early_stopping_rounds=100, eval_metric="auc",gamma=0.9274165460731112,subsample=0.9676205936982302,
                        reg_alpha=0.7856183692190426,reg_lambda=3.821155642978947,
                        colsample_bytree=round(0.7326425043432427, 5) 
                       )


# evaluation metric을 auc로, early stopping은 100 으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)])


xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:,1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))

[0]	validation_0-auc:0.81733	validation_1-auc:0.81347
[1]	validation_0-auc:0.83016	validation_1-auc:0.82807
[2]	validation_0-auc:0.83533	validation_1-auc:0.83045
[3]	validation_0-auc:0.83905	validation_1-auc:0.83540
[4]	validation_0-auc:0.84180	validation_1-auc:0.83722
[5]	validation_0-auc:0.84313	validation_1-auc:0.83763
[6]	validation_0-auc:0.84494	validation_1-auc:0.83979
[7]	validation_0-auc:0.84640	validation_1-auc:0.84063
[8]	validation_0-auc:0.84749	validation_1-auc:0.84101
[9]	validation_0-auc:0.84834	validation_1-auc:0.84221
[10]	validation_0-auc:0.85008	validation_1-auc:0.84322
[11]	validation_0-auc:0.85081	validation_1-auc:0.84344
[12]	validation_0-auc:0.85282	validation_1-auc:0.84372
[13]	validation_0-auc:0.85348	validation_1-auc:0.84447
[14]	validation_0-auc:0.85453	validation_1-auc:0.84566
[15]	validation_0-auc:0.85530	validation_1-auc:0.84608
[16]	validation_0-auc:0.85589	validation_1-auc:0.84624
[17]	validation_0-auc:0.85630	validation_1-auc:0.84697
[18]	validation_0-au

In [15]:
# from lightgbm import LGBMClassifier,early_stopping, log_evaluation
# from sklearn.metrics import roc_auc_score

# lgbm_clf = LGBMClassifier(n_estimators=500)

# eval_set=[(X_tr, y_tr), (X_val, y_val)]
# lgbm_clf.fit(
#     X_tr, y_tr, 
#     eval_metric='auc',
#     eval_set=eval_set,
#     callbacks=[
#         early_stopping(stopping_rounds=100),
#         log_evaluation(period=50)
#     ])


# lgbm_roc_score = roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:,1])
# print(f'ROC AUC: {lgbm_roc_score:.4f}')

In [16]:
# from lightgbm import LGBMClassifier, early_stopping, log_evaluation
# from sklearn.metrics import roc_auc_score

# lgbm_clf = LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     random_state=0,
#     n_jobs=-1,
#     verbosity=-1
# )
# lgbm_clf.fit(
#     X_tr,
#     y_tr,
#     eval_set=[(X_val, y_val)],
#     eval_metric='auc',
#     callbacks=[
#         early_stopping(stopping_rounds=100),
#         log_evaluation(period=0)
#     ]
# )


# # 확률값 예측
# pred_probs = lgbm_clf.predict_proba(X_test)[:, 1]

# # ROC-AUC
# lgbm_auc = roc_auc_score(y_test, pred_probs)

# print("LGBM ROC-AUC:", lgbm_auc)
# print("Best iteration:", lgbm_clf.best_iteration_)

In [17]:
# hyperopt
# hyperopt 하이퍼 파라미터 튜닝

# from hyperopt import hp
# # 1. search space 선언
# lgbm_search_space = {
#     'num_leaves': hp.quniform('num_leaves', 10, 60, 1),
#     'max_depth': hp.quniform('max_depth', 4, 15, 1),
#     'min_child_samples': hp.quniform('min_child_samples', 60, 200, 1),
#     'subsample': hp.uniform('subsample', 0.6, 1),
#     'learning_rate': hp.uniform('learning_rate', 0.01, 0.3)
# }


In [18]:
# from sklearn.model_selection import KFold
# from sklearn.metrics import roc_auc_score

# def objective_func(search_space):
#     lgbm_clf =  LGBMClassifier(n_estimators=100, num_leaves=int(search_space['num_leaves']),
#                                max_depth=int(search_space['max_depth']),
#                                min_child_samples=int(search_space['min_child_samples']),
#                                subsample=search_space['subsample'],
#                                learning_rate=search_space['learning_rate'])
#     # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
#     roc_auc_list = []
   
#     # 3개 k-fold방식 적용
#     kf = KFold(n_splits=3)
#     # X_train을 다시 학습과 검증용 데이터로 분리
#     for tr_index, val_index in kf.split(X_train):
#         # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
#         X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
#         X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]


#         # early stopping은 50회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
#         lgbm_clf.fit(X_tr, y_tr, callbacks=[early_stopping(50)], eval_metric="auc",
#            eval_set=[(X_tr, y_tr), (X_val, y_val)])


#         # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
#         score = roc_auc_score(y_val, lgbm_clf.predict_proba(X_val)[:, 1])
#         roc_auc_list.append(score)
   
#     # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
#     # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
#     return -1*np.mean(roc_auc_list)


In [19]:
# from hyperopt import fmin, tpe, Trials


# trials = Trials()


# # fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.
# best = fmin(fn=objective_func, space=lgbm_search_space, algo=tpe.suggest,
#             max_evals=200, # 최대 반복 횟수를 지정합니다.
#             trials=trials, rstate=np.random.default_rng(seed=30))


# print('best:', best)

In [20]:

# from lightgbm import LGBMClassifier, early_stopping, log_evaluation
# from sklearn.metrics import roc_auc_score

# lgbm_clf = LGBMClassifier(
#     n_estimators=500,
#     num_leaves=int(best['num_leaves']),
#     max_depth=int(best['max_depth']),
#     min_child_samples=int(best['min_child_samples']),
#     subsample=round(best['subsample'], 5),
#     learning_rate=round(best['learning_rate'], 5),
#     random_state=0,
#     n_jobs=-1,
#     verbosity=-1
# )
# lgbm_clf.fit(
#     X_tr,
#     y_tr,
#     eval_set=[(X_val, y_val)],
#     eval_metric='auc',
#     callbacks=[
#         early_stopping(stopping_rounds=100),
#         log_evaluation(period=0)
#     ]
# )


# # 확률값 예측
# pred_probs = lgbm_clf.predict_proba(X_test)[:, 1]

# # ROC-AUC
# lgbm_auc = roc_auc_score(y_test, pred_probs)

# print("LGBM ROC-AUC:", lgbm_auc)
# print("Best iteration:", lgbm_clf.best_iteration_)

In [21]:
#LGBM전용 (핏하고 사용하는 방법)

# feat_imp = pd.Series(
#     lgbm_clf.feature_importances_,
#     index=X_tr.columns
# )

# zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()

# X_tr = X_tr.drop(columns=zero_imp_cols)
# X_val = X_val.drop(columns=zero_imp_cols)
# X_test = X_test.drop(columns=zero_imp_cols)

# print(f"제거된 중요도 0인 컬럼 수: {len(zero_imp_cols)}")